In [ ]:
# Sentence Length
import nltk
from nltk.tokenize import sent_tokenize
import pandas as pd

# Load NLTK resources
nltk.download('punkt')

# Function to calculate sentence count
def calculate_sentence_count(text):
    sentences = sent_tokenize(text)
    return len(sentences)

# Assuming your DataFrame is already loaded, let's assume it's called df
# Calculate sentence count for each essay
df['sentence_count'] = df['full_text'].apply(calculate_sentence_count)

plt.figure(figsize=(10, 6))
plt.scatter(df['sentence_count'], df['score'], color='blue')
plt.title('Score vs Sentence Length')
plt.xlabel('Essay Length')
plt.ylabel('Score')
plt.grid(True)
plt.show()

In [ ]:
# Calculate the median essay length by score
median_lengths_by_score = df.groupby('score')['essay_length'].median().reset_index()

# Merge this median length back to the original dataframe to use as a feature
df = df.merge(median_lengths_by_score, on='score', suffixes=('', '_median'))

df

The plot above shows that there is an evident relationship between the scores and the lengths of the essays. Therefore, we are choosing to median essay length as a feature and seeing what the results look like.

In [ ]:
median_lengths_by_score

In [ ]:
from sklearn.model_selection import train_test_split

# Edward Train

# # Define features and target
# X = df[['essay_length_median']]  # Features
# y = df['score']  # Target

# # Split the data into training and testing sets
# X_train, X_val, y_train, y_val = train_test_split(X, y, shuffle = True, random_state = 100000)
# X_train = X_train.fillna(0)

# Sam Train

from gensim.models import Word2Vec

word2vec_model = Word2Vec(sentences=df['processed_text'], vector_size=100, window=5, min_count=1, workers=4)

X = df['processed_text']
y = df['score']

def get_average_embedding(text):
    words = comprehensive_text_preprocessing(text)
    # Filter out words that are not in the vocabulary of the Word2Vec model
    words_in_vocab = [word for word in words if word in word2vec_model.wv]
    if len(words_in_vocab) > 0:
        return np.mean([word2vec_model.wv[word] for word in words_in_vocab], axis=0)
    else:
        return np.zeros(word2vec_model.vector_size)  # Return zero vector if no words in vocabulary

X_features = X.apply(get_average_embedding)

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X_features.tolist(), y, test_size=0.2, random_state=42)

# Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error


# Edward's Model

# # Initialize and train the linear regression model
# lin_reg = LinearRegression()
# lin_reg.fit(X_train, y_train)

# # Make predictions and evaluate
# y_pred_lin = lin_reg.predict(X_val)
# mse_lin = mean_squared_error(y_val, y_pred_lin)
# print(f'Linear Regression MSE: {mse_lin}')

# Sam's Model

# Train linear regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict scores on test set
y_pred = model.predict(X_test)

# Evaluate model
mse = mean_squared_error(y_test, y_pred)
print("Mean Squared Error:", mse)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

test_df = pd.read_csv("/kaggle/input/learning-agency-lab-automated-essay-scoring-2/test.csv")

# Edward PreProcess

# # ---- PreProcessing ----

test_df['processed_text'] = test_df['full_text'].apply(comprehensive_text_preprocessing)
# test_df['essay_length'] = test_df['processed_text'].apply(len)

# # # Calculate the median essay length by score
# median_lengths_by_score = test_df.groupby('essay_id')['essay_length'].median().reset_index()

# # # Merge this median length back to the original dataframe to use as a feature
# test_df = test_df.merge(median_lengths_by_score, on='essay_id', suffixes=('', '_median'))

# # ---- PreProcessing End ----

from gensim.models import Word2Vec

X = test_df['processed_text']

def get_average_embedding(text):
    words = comprehensive_text_preprocessing(text)
    # Filter out words that are not in the vocabulary of the Word2Vec model
    words_in_vocab = [word for word in words if word in word2vec_model.wv]
    if len(words_in_vocab) > 0:
        return np.mean([word2vec_model.wv[word] for word in words_in_vocab], axis=0)
    else:
        return np.zeros(word2vec_model.vector_size)  # Return zero vector if no words in vocabulary

X_features = X.apply(get_average_embedding)

# Make predictions on the test set
y_pred_lin = model.predict(X_features.tolist())

# Create a DataFrame to store essay_id and predicted scores
predictions_df = pd.DataFrame({'essay_id': test_df['essay_id'], 'score': [int(np.round(x)) for x in y_pred_lin]})

# Output predictions to a CSV file
predictions_df.to_csv("/kaggle/working/submission.csv", index=None)